# Thinking-Budget Qwen-1.5B — does the post-training do anything?

Runs the base model and the LoRA adapter **side by side on the same prompts** so the
comparison is apples-to-apples. Free Colab T4 is enough (1.5B in fp16 ≈ 3.5 GB).

**Runtime → Change runtime type → T4 GPU** before you start.

Two things to look for, and they are not the same thing:

1. **Compression (the result that worked).** The adapter answers in ~3x fewer tokens.
2. **The budget dial (the result that did not).** Output length barely moves with the
   requested `N`. The sweep below is built to show that honestly rather than hide it.


In [ ]:
!pip -q install "transformers>=4.44" "peft>=0.11" accelerate
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

## Load base + adapter once

`PeftModel` keeps one set of base weights; `disable_adapter()` toggles the LoRA off, so
"base" and "trained" are the same weights with and without the delta — no second download.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
ADAPTER = "rk9595/thinking-budget-qwen1.5b-lcpo"

tok = AutoTokenizer.from_pretrained(BASE)
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = PeftModel.from_pretrained(
    AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float16, device_map="cuda"),
    ADAPTER,
    subfolder="adapter",
)
model.eval()
print("loaded")

## Prompt format

This has to match training exactly. The wording is **`exactly N tokens`** — the LCPO-Exact
run was trained on that string. `maximum` was the superseded Max run and gives you a
model-prompt mismatch.

In [ ]:
INSTR = "Let's think step by step and output the final answer within \\boxed{}."

def build(problem, budget):
    content = f"{problem}\n\n{INSTR} Think for exactly {budget} tokens."
    return tok.apply_chat_template([{"role": "user", "content": content}],
                                   tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate(problems, budget, use_adapter, max_new_tokens=4096):
    """Returns [(text, n_generated_tokens)] for a batch of problems."""
    prompts = [build(p, budget) for p in problems]
    enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")

    def run():
        return model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=True,
                              temperature=0.6, top_p=0.95, pad_token_id=tok.pad_token_id)

    if use_adapter:
        out = run()
    else:
        with model.disable_adapter():
            out = run()

    gen = out[:, enc.input_ids.shape[1]:]
    results = []
    for row in gen:
        ids = row.tolist()
        # pad_token may alias eos, so trim at the first eos instead of counting
        # non-pad ids. Keeping the eos matches how vLLM counted during eval.
        if tok.eos_token_id in ids:
            ids = ids[: ids.index(tok.eos_token_id) + 1]
        results.append((tok.decode(ids, skip_special_tokens=True), len(ids)))
    return results

## 1. One problem, side by side

The screenshot-worthy one: same question, same budget, base vs trained.

In [ ]:
PROBLEM = ("What is the smallest positive integer $n$ such that $n^2$ is divisible by 18 "
           "and $n^3$ is divisible by 640?")
BUDGET = 512

(base_text, base_n),    = generate([PROBLEM], BUDGET, use_adapter=False)
(tuned_text, tuned_n),  = generate([PROBLEM], BUDGET, use_adapter=True)

print(f"BASE     {base_n:>5} tokens")
print(f"TRAINED  {tuned_n:>5} tokens     ({base_n/max(tuned_n,1):.1f}x shorter)")
print("=" * 78)
print("BASE:\n"); print(base_text)
print("=" * 78)
print("TRAINED:\n"); print(tuned_text)

## 2. Budget sweep — the honest test

Same problems at four budgets. If the dial worked, the trained line would climb with the
budget. It does not: the paper result is 781/783/763/821 tokens for 256/512/1024/2048, a
**1.08x spread**. A handful of samples is noisy, so treat this as a sanity check of that
number, not a re-measurement of it.

~10-15 min on a T4. Raise `N_PROBLEMS` if you want tighter error bars and have time.

In [ ]:
PROBLEMS = [
    "What is $17 \\times 23$?",
    "If $3x + 7 = 22$, what is $x$?",
    "A bag has 4 red and 6 blue marbles. Two are drawn without replacement. "
    "What is the probability both are red?",
    "What is the sum of all positive divisors of 60?",
]
BUDGETS = [256, 512, 1024, 2048]
N_PROBLEMS = 4

import statistics, time
rows = []
for budget in BUDGETS:
    for label, use_adapter in [("base", False), ("trained", True)]:
        t0 = time.time()
        outs = generate(PROBLEMS[:N_PROBLEMS], budget, use_adapter=use_adapter)
        lengths = [n for _, n in outs]
        rows.append({"budget": budget, "model": label,
                     "mean_tokens": statistics.mean(lengths), "lengths": lengths})
        print(f"budget {budget:>5}  {label:<8} mean {statistics.mean(lengths):>7.0f} tokens "
              f"({time.time()-t0:.0f}s)")

In [ ]:
print("| budget | base tokens | trained tokens | compression |")
print("|---|---|---|---|")
for b in BUDGETS:
    base = next(r for r in rows if r["budget"] == b and r["model"] == "base")["mean_tokens"]
    tuned = next(r for r in rows if r["budget"] == b and r["model"] == "trained")["mean_tokens"]
    print(f"| {b} | {base:.0f} | {tuned:.0f} | {base/max(tuned,1):.1f}x |")

tuned_means = [next(r for r in rows if r["budget"] == b and r["model"] == "trained")["mean_tokens"]
               for b in BUDGETS]
print(f"\ntrained spread across budgets: {max(tuned_means)/min(tuned_means):.2f}x "
      f"(1.00x = fully inert dial, paper value 1.08x)")

## 3. The chart

Two lines. The gap between them is what the post-training bought. The flatness of the
orange line is what it did not.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.5))
for label, color in [("base", "#888888"), ("trained", "#d95f02")]:
    ys = [next(r for r in rows if r["budget"] == b and r["model"] == label)["mean_tokens"]
          for b in BUDGETS]
    ax.plot(BUDGETS, ys, "o-", color=color, label=label, linewidth=2)

ax.plot(BUDGETS, BUDGETS, "--", color="#bbbbbb", linewidth=1, label="perfect budget adherence")
ax.set(xlabel="requested budget N (tokens)", ylabel="mean tokens generated",
       title="Thinking-Budget Qwen-1.5B: compression yes, dial no")
ax.set_xscale("log", base=2); ax.set_xticks(BUDGETS)
ax.get_xaxis().set_major_formatter(plt.ScalarFormatter())
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("budget_demo.png", dpi=150)
plt.show()

## What this is actually good for

Use it when you want **the same answer in a third of the tokens** — a cost/latency win on
competition-math-shaped prompts, at −2.0 accuracy points on MATH-500 (−5.2 on GSM8K, which
is already terse and has less fat to cut).

Do **not** use it as a test-time-compute dial. The `N` in the prompt is close to inert;
that experiment failed three times and the model card says so.
